In [1]:
# This notebook demonstrates how to use ROSALIA to analyze stray light in a simple exposure. 
import rosalia as rs
import numpy as np
import rosalia as rs
import matplotlib.pyplot as plt# os.system("cp -v /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933_BAK/MAST/Roman/r0102001002001005001_0004_wfi*.asdf /Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/" )
from matplotlib.colors import SymLogNorm

#path = "/Users/aborlaff/NASA/ROSALIA_DEPOT/MAST_2026-09-16T0933/MAST/Roman/r0102001002001005001_0004_wfi*_f158_cal.asdf"
#custom_roman_exposure = rs.core.exposure(filename=path) 
# straylight_out = custom_roman_exposure.straylight(verbose=True)



@Alex Borlaff the Dragon's breath was observed in all filters over the past few days in program 1043, execution plan 3, passes 2 & 3 at the bottom of WFI18 (edited)

In [2]:
"""
Example of criteria for a roman_query using astroquery.mast.query_criteria
search = {'program': 1020,
         'observation': 5,
         'pass': 2, 
         #'exposure_start_time': '2026-09-15T02:01:47.4080000',
         'product_type': 'l2', 
         'detector': 'WFI06',
         # 'optical_element': 'F158',
         'exposure_type': "WFI_IMAGE"}

Here we use a region criteria, based on a set of coordinates and a search radius.
"""

search = {'program': 1043,
         'product_type': 'l2', 
         'pass': 2, 
         'detector': 'WFI18',
         #'optical_element': 'F146',
         #'exposure_start_time': '2026-09-28',
         'exposure_type': "WFI_IMAGE"}

results, products, exposures, downloaded = rs.mast.query.roman_query(criteria=search, filter="F146", mindate='2026-09-28T06:08:04.331', file_suffix="_cal", download_products=False)

INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]
Mission: roman
Service: search
INFO: 59 of 104 products were duplicates. Only returning 45 unique product(s). [astroquery.mast.utils]
INFO: To return all products, use `MastMissions.get_product_list` [astroquery.mast.missions]


In [3]:
data_stream = rs.mast.query.stream_roman_mast(products=products, row=0)

INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]


In [4]:
roman_exposure = rs.core.exposure(data_stream)

TEMP WARNING: RA_TARG is set to element 0 - This has to be fixed on MAST
TEMP WARNING: DEC_TARG is set to element 0 - This has to be fixed on MAST
TEMP WARNING: PA is set to element 0 - This has to be fixed on MAST


In [11]:
roman_exposure.DEC_TARG

46.38833981445026

In [ ]:
roman_exposure.DEC_TARG

In [8]:
roman_exposure.straylight()

2026-09-30T10:52:12.307581 > Fetching source catalog
INFO: radius parameter (minimum distance to search for individual stars) is > 0.5 degrees.
Gaia/2MASS/WISE query database can take several minutes to process. Please be patient.
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⣯In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
INFO: Retrieving tables... [astroquery.utils.tap.core]
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⣽INFO: Parsing tables... [astroquery.utils.tap.core]
Querying Gaia/2MASS/WISE/JPL Horizons databases. This might take a few minutes... ⢿INFO: Done. [astroquery.utils.tap.core]
Querying 

1it [00:00,  3.15it/s]

2026-09-30T10:52:44.702684 > Done.
2026-09-30T10:52:44.711432 > Starting Stray-light scan: 



  0%|          | 0/1 [00:03<?, ?it/s]


TypeError: bad operand type for unary -: 'list'

In [ ]:
roman_exposure.zodiacal(zody_mode="stsci")

AttributeError: 'list' object has no attribute 'replace'

In [ ]:
# Quick aperture photometry 
# dragon_flux_sum = 520495.06 # DN/s	2.23559
# background_sum = 347943.57	# DN/s 	1.59042

In [ ]:
custom_roman_exposure.SCIEXTS

In [ ]:
# Index 3 holds the affected exposure. 
image = custom_roman_exposure.DATA[3]
sky = np.nanmedian(custom_roman_exposure.DATA[3])
skycor = image[0:1500, 500:1500] - sky

fig, ax = plt.subplots(figsize=(8,8))
im = ax.imshow(skycor, origin="lower",
               norm=SymLogNorm(linthresh=1, vmin=-0.2, vmax=20))
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)      
ax.set_title("r0102001002001005001_0004 F158 WFI06", fontsize=10)
plt.tight_layout()
plt.savefig("r0102001002001005001_0004_F158_WFI06.png", dpi=300)


In [ ]:
# Make a radial profile
# First save the image to a fits file
drz_data, drz_wcs, outname = custom_roman_exposure.save_drz(outname="dragon_snap.fits",  
                               data_list=[custom_roman_exposure.DATA[3]-sky], 
                               wcs_list=[custom_roman_exposure.ASTROPYWCS[3]], resolution=0.11)



In [ ]:
ra_dragon_center = 229.6405032
dec_dragon_center = -3.5874811
# radial_mask = rs.utils.create_radial_mask(xsize=image.shape[0], ysize=image.shape[1], q=0.5, theta=-45, center=[2549.2, 4867.4], radius=None)
rbins = np.linspace(0, 2000, 250)

xcen = 2551
ycen = 4856
pa=22
q = 0.25
profile = rs.utils.make_profile(image="dragon_snap.fits", ext=0, xcen=2551, ycen=4856, rbins=rbins,
                                q=q, theta=pa)

rs.utils.save_fits(profile["model"], "radial_mask.fits")



In [ ]:
fig, ax = plt.subplots(figsize=(8,8))


ysnap = 4000
xsnap = 2200
width = 1000
im = ax.imshow(drz_data[ysnap:ysnap+width,xsnap:xsnap+width], origin="lower", norm=SymLogNorm(linthresh=1, vmin=-0.2, vmax=20))
from matplotlib.patches import Ellipse
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r'Flux (DN/s)')      
ax.set_title("r0102001002001005001_0004 F158 WFI06", fontsize=14)

radius = np.linspace(0,1500,25)
for r in radius:
    ellipse = Ellipse(xy=(xcen-xsnap, ycen-ysnap), width=q*r, height=r, angle=pa, alpha=0.5,
                    edgecolor='white', linestyle='--', facecolor='none', linewidth=1)
    ax.add_patch(ellipse)

ax.set_xlabel("X (pixel)")
ax.set_ylabel("Y (pixel)")
plt.savefig("r0102001002001005001_0004_Dragon_ellipses.png", dpi=300)
plt.show()


In [ ]:
# profile["model"].shape
plt.imshow(drz_data, origin="lower", vmin=0, vmax=1)
plt.show()
plt.imshow(profile["model"], origin="lower", vmin=0, vmax=1)
plt.show()
plt.imshow(drz_data - profile["model"], origin="lower", vmin=0, vmax=1)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.plot(profile["profile"]["r"], profile["profile"]["int"], linewidth=2, color="dodgerblue",
        label="r0102001002001005001_0004 \n F158 WFI06 Dragon's breath")
ax.fill_between(x=profile["profile"]["r"], 
                y1=profile["profile"]["int_s1down"],
                y2=profile["profile"]["int_s1up"], alpha=0.2, color="dodgerblue")

ax.set_ylim(0, 15)
ax.set_xlim(0,500)
ax.set_ylabel("Surface brightness profile (DN/s)")
ax.set_xlabel("Radius (pixel)")
ax.legend()
plt.tight_layout()
plt.savefig("r0102001002001005001_0004_Dragon_profile.png", dpi=300)

In [ ]:
profile["profile"]["r"]